In [1]:
import os
import time
import json
import pickle
import pandas as pd
import numpy as np
from functools import partial
import joblib

from datetime import datetime
from tqdm import tqdm
from dotenv import load_dotenv
from pathlib import Path

In [2]:
from text2graphapi.src.IntegratedSyntacticGraph import ISG
import networkx as nx
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import torch

import optuna
import mlflow
from databricks.sdk import WorkspaceClient

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


2025-12-19 13:38:11,186; - DEBUG; - Import libraries/modules from :PROD


In [3]:
import seaborn as sns
import matplotlib.pyplot as plt

Define path variables

In [4]:
representation_type = "integrated_syntactic_graph"
developer_initials = "JP"

In [5]:
current_dir = Path.cwd()
env_path = current_dir.parent.parent / "conf" / "local" / ".env"
results_path = current_dir.parent.parent / "results" / "graph"

train_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-training-large-cleaned.jsonl"
validation_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-validation-large-cleaned.jsonl"
test_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan21-authorship-verification-test-cleaned.jsonl"

Connect to databricks for logging results

In [6]:
load_dotenv(env_path)

w = WorkspaceClient()   
print("Connected to:", w.config.host)

mlflow.set_tracking_uri("databricks")
mlflow.autolog()

2025/12/19 13:38:12 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.4.0 <= scikit-learn, but the installed version is 1.3.2. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2025/12/19 13:38:12 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/12/19 13:38:12 WARNING mlflow.utils.autologging_utils: MLflow statsmodels autologging is known to be compatible with 0.14.1 <= statsmodels, but the installed version is 0.14.0. If you encounter errors during autologging, try upgrading / downgrading statsmodels to a compatible version, or try upgrading MLflow.


Connected to: https://dbc-1ea3ad0e-f504.cloud.databricks.com


2025/12/19 13:38:12 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.
2025/12/19 13:38:12 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.


What are GPU are the experiments run on

In [7]:
!nvidia-smi

Fri Dec 19 13:38:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10                     Off |   00000000:61:00.0 Off |                    0 |
|  0%   50C    P8             25W /  150W |       3MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
running_on_gpu = torch.cuda.is_available()

In [9]:
if running_on_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    gpu_vram_gb = round(gpu_props.total_memory / (1024**3), 2)
else:
    gpu_name = "CPU"
    gpu_props = "N/A"
    gpu_vram_gb = 0

Empty the GPU from previous experiments

In [10]:
import gc
import torch
torch.cuda.empty_cache()
gc.collect()

89

In [11]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

Classification threshold constant specification

In [12]:
classification_thresholds = [x/1000 for x in range(200, 999)]

# Load dataset

#### Load training data

In [13]:
train_data_file_size = os.path.getsize(train_data_full_cleaned_path)
train_data = []

with open(train_data_full_cleaned_path, 'r') as f:
    with tqdm(total=train_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            train_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(train_data)} items.")

Loading data: 100%|██████████| 11.3G/11.3G [00:22<00:00, 512MB/s]


Successfully loaded 273301 items.


In [14]:
train_data_df = pd.DataFrame(train_data)

Prepare dataset for cosine embedding loss

In [15]:
train_data_df.head(10)

,id,pair,same
0,e05b9c0b-88a1-5608-b7e8-ab1fc6b78dc1,"[Well, ever since you and Kurt broke up youve ...",False
1,12f73a20-cdf3-58df-b5bb-9392eec9b486,"[The thing is, Ryouga has no reason to run aft...",False
2,d82c6764-451b-544c-8711-c139e9349c56,"[Ehhhh nah, its silly' Its my job to listen to...",True
3,876b8380-9260-5427-93e8-dc31155c3edd,"[Glaring at the arrogant spark, Always asks va...",False
4,357e8471-35b9-50b4-9ac0-286ac0e8b101,[Runa limped across the small space to an open...,False
5,6b39fe22-409f-5329-9fe1-ddf4a70bedd1,"[And thats retired Commander, if you please St...",False
6,76ed2017-0c9f-580f-83b1-5a041a8169ec,[Meet you downstairs in twenty minutes I say w...,True
7,fd8deb2e-06de-5c92-9a4c-927891c53657,"[Since Ive seen so many others do so, Im going...",False
8,3ce5e811-a57c-5fbf-9f9a-2ee636e50be6,[party Eishi exclaimed Omi shook his head and ...,False
9,25a17cd2-6b01-5fba-99ef-e631e56e181d,[After a few moments she found that Red was ri...,False


#### Create a subset of training data for finetuning

In [31]:
train_tuning_size = 999
train_tuning_data_df = train_data_df.sample(n=train_tuning_size, random_state=42)
train_tuning_data_df = train_tuning_data_df.reset_index(drop=True)
train_tuning_data_df.head()

,id,pair,same
0,1c4a05b6-dabb-5a9d-9e6a-7709b07d8dff,[comes home at about onethirty that night Case...,True
1,aff2cdfb-625d-5fa7-acb8-a0ff9876b34f,"[He rubbed his tired, gritty eyes The clock on...",False
2,15fc44d2-e1c9-574b-9fa3-fdb29f20cc68,"[Mikado grinned and explained, no longer using...",False
3,e77681c2-8d90-5cac-8ab0-07d7c9277670,[Thanks he smiled Now to write Characters and ...,False
4,8c38b587-f54c-5e52-94cb-abb48ce14c5a,"[soldierscare that ended earlier, and all were...",True


#### Load validation data

In [32]:
val_data_file_size = os.path.getsize(validation_data_full_cleaned_path)
val_data = []

with open(validation_data_full_cleaned_path, 'r') as f:
    with tqdm(total=val_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            val_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(val_data)} items.")

Loading data: 100%|██████████| 103M/103M [00:01<00:00, 85.6MB/s] 


Successfully loaded 2500 items.


In [33]:
val_data_df = pd.DataFrame(val_data)

#### Load testing data

In [34]:
test_data_file_size = os.path.getsize(test_data_full_cleaned_path)
test_data = []

with open(test_data_full_cleaned_path, 'r') as f:
    with tqdm(total=test_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            test_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(test_data)} items.")

Loading data: 100%|██████████| 826M/826M [00:01<00:00, 576MB/s] 


Successfully loaded 19999 items.


In [35]:
test_data_df = pd.DataFrame(test_data)

In [61]:
train_tuning_data_df = train_tuning_data_df.head()
val_data_df = val_data_df.head()

# Build graphs and extract features

Parse ISG's nodes POS and lemma function

In [153]:
def parse_node(node):
    s = str(node)
    if "_" in s:
        lemma, pos = s.rsplit("_", 1)
        return lemma.lower(), pos
    return s.lower(), None

Parse dependency

In [154]:
def parse_dependency(data):
    dep = data.get("gramm_relation")
    if dep is None:
        return None
    return dep.split("_", 1)[0]

Extract multi-level features from graph

In [214]:
def extract_multilevel_features_from_isg(graph):
    graph_object = graph["graph"]
    features = Counter()

    # lexical + morphological 
    for node in graph_object.nodes:
        lemma, pos = parse_node(node)
        features[f"LEX::{lemma}"] += 1
        
        if pos:
            features[f"POS::{pos}"] += 1

    # syntactic
    for _, _, data in graph_object.edges(data=True):
        dependency = parse_dependency(data)
        if dependency:
            features[f"DEP::{dependency}"] += 1

    return features

Convert features to numeric vectors function

In [ ]:
def features_to_matrix(counters):
    vocab = sorted(set().union(*counters))
    index = {f: i for i, f in enumerate(vocab)}

    X = np.zeros((len(counters), len(vocab)))
    for i, c in enumerate(counters):
        for f, v in c.items():
            X[i, index[f]] = v

    return X, vocab

Format the texts in format required by text2graphapi

In [215]:
def texts_to_isg(isg, texts):
    corpus_docs = [
        {"id": i, "doc": text}
        for i, text in enumerate(texts)
    ]
    return isg.transform(corpus_docs)

Convert texts to text2graphapi integrated syntactic graphs

In [216]:
def build_isg_features(train_df, test_df):
    
    isg = ISG(
        graph_type="DiGraph",
        language="en",
        apply_prep=True,
        output_format="networkx"
    )
    start = time.perf_counter()
    train_texts1 = train_df["pair"].apply(lambda x: x[0])

    train_texts2 = train_df["pair"].apply(lambda x: x[1])
    test_texts1 = test_df["pair"].apply(lambda x: x[0])
    test_texts2 = test_df["pair"].apply(lambda x: x[1])
    
    print("Converting to graphs \n")
    X_train1 = texts_to_isg(isg, train_texts1)
    
    end = time.perf_counter()
    print(f"Execution time: {end - start:.6f} seconds")
    
    start = time.perf_counter()
    C_train1 = [extract_multilevel_features_from_isg(g) for g in X_train1]
    print(C_train1[0])
    end = time.perf_counter()
    print(f"Execution time: {end - start:.6f} seconds")
    
    X_train2 = texts_to_isg(isg, train_texts2)
    
    X_test1 = texts_to_isg(isg, test_texts1)
    X_test2 = texts_to_isg(isg, test_texts2)
    

    X_train = (X_train1, X_train2)
    X_test = (X_test1, X_test2)
    
    return X_train, X_test

# Set up evaluation functions

Evaluate model function

In [217]:
def compute_classifier_scores(test_data_df, model):
    start_time = time.time()
    verification_results = []
    model.eval()

    with torch.no_grad():
        for i in tqdm(test_data_df.index, desc="Processing rows"):
            
            resulting_df_row = {}
            resulting_df_row['id']  = test_data_df.loc[i, 'id']
            resulting_df_row['actual_result'] = test_data_df.loc[i, 'same']
            
            text1 = test_data_df.loc[i, 'pair'][0]
            text2 = test_data_df.loc[i, 'pair'][1]

            enc = tokenizer(
                text1,
                text2,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=8192
            ).to(device)

            logits = model(**enc).logits
            probs = F.softmax(logits, dim=-1)
            prob_same = probs[0, 1].item()

            result_df_row["propability_same_author"]
            verification_results.append(resulting_df_row)

    result_df = pd.DataFrame(verification_results)
    print("--- Execution Time: %s seconds ---" % round(time.time() - start_time, 2))

    return result_df

Evaluation function

In [218]:
def evaluate_results(y_true, y_pred, average='binary'):
    accuracy = round(accuracy_score(y_true, y_pred)*100, 2)
    precision = round(precision_score(y_true, y_pred, average=average)*100, 2)
    recall = round(recall_score(y_true, y_pred, average=average)*100, 2)
    f1 = round(f1_score(y_true, y_pred, average=average)*100, 2)
    return accuracy, precision, recall, f1

Optimal threshold search

In [219]:
def evaluate_classification_thresholds(result_df, classification_thresholds):
    results = []
    y_embeddings = result_df['propability_same_author']
    y_true = result_df['actual_result']
    for threshold in thresholds:
        y_pred = (result_df["propability_same_author"] >= threshold).astype(int)
        accuracy, precision, recall, f1 = evaluate_results(y_true, y_pred)
        results.append({
            "threshold": threshold,
            "accuracy" : accuracy,
            "precision" : precision,
            "recall" : recall,
            "f1" : f1
        })
    return pd.DataFrame(results)

Create histogram of F1 score for different thresholds

In [220]:
def plot_f1_vs_threshold(results_df):
    plt.figure(figsize=(9, 5))
    plt.plot(results_df["threshold"], results_df["f1"], linewidth=2)
    plt.xlabel("Threshold")
    plt.ylabel("F1 Score")
    plt.title("F1 Score vs Classification Threshold")
    plt.grid(True)
    plt.tight_layout()
    return plt.gcf()

Create the confusion matrix

In [221]:
def create_confusion_matrix(y_true, y_pred, labels=[False, True]):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    cm_fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, 
                fmt='d', 
                cmap='Blues',
                xticklabels=labels, 
                yticklabels=labels, 
                ax=ax,
                cbar=False)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix: {model_id}')
    return cm

# Find the top threshold

result_df = compute_similarities(train_tuning_data_df, model)
result_df.head()

In [222]:
X_train, X_test = build_isg_features(train_tuning_data_df, val_data_df)

2025-12-19 16:56:24,642; - INFO; - Has already installed spacy model en_core_web_sm
Converting to graphs 

2025-12-19 16:56:24,897; - INFO; - Init transformations: Text to Integrated Syntactic Graphs
2025-12-19 16:56:24,897; - INFO; - Transforming 5 text documents...
2025-12-19 16:56:25,767; - INFO; - Done transformations
Execution time: 0.873592 seconds
0
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
NOUN
N